# Train AQI model

This notebook is self-contained: it loads the raw data, cleans it, trains a model, reports holdout metrics, and saves the fitted pipeline to `models/random_forest_aqi.pkl`. Run all cells from top to bottom.


In [ ]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/raw/city_day.csv").exists())
df = pd.read_csv(ROOT / "data/raw/city_day.csv")


In [ ]:
# Clean the source data
if "Date" in df:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.sort_values(["City", "Date"] if "Date" in df else ["City"]).copy()
pollutants = [c for c in ["PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3", "Benzene", "Toluene", "Xylene"] if c in df]
if pollutants:
    df[pollutants] = df.groupby("City", sort=False)[pollutants].transform(lambda values: values.interpolate(limit_direction="both"))
    for col in pollutants:
        df[col] = df[col].fillna(df.groupby("City")[col].transform("median"))
df = df.dropna(subset=["AQI"]).drop_duplicates()
if {"City", "Date"}.issubset(df.columns):
    df = df.drop_duplicates(["City", "Date"], keep="first")
for col in pollutants + ["AQI"]:
    if col in df:
        lower, upper = df[col].quantile([0.01, 0.99])
        df[col] = df[col].clip(lower, upper)


In [ ]:
# Use known predictors; exclude raw date and any target-derived values to avoid leakage.
feature_columns = [c for c in df.columns if c not in {"AQI", "AQI_Bucket", "Date"}]
categorical = [c for c in feature_columns if df[c].dtype == "object"]
numeric = [c for c in feature_columns if c not in categorical]
preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median", add_indicator=True), numeric),
    ("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical),
])
model = Pipeline([("preprocess", preprocessor), ("regressor", RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=4))])
X_train, X_test, y_train, y_test = train_test_split(df[feature_columns], df["AQI"], test_size=0.2, random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
metrics = {"MAE": mean_absolute_error(y_test, predictions), "RMSE": mean_squared_error(y_test, predictions) ** 0.5, "R2": r2_score(y_test, predictions)}
print(metrics)


In [ ]:
# Refit on all labeled records and save a self-contained model artifact.
model.fit(df[feature_columns], df["AQI"])
model_dir = ROOT / "models"
model_dir.mkdir(parents=True, exist_ok=True)
joblib.dump({"model": model, "features": feature_columns}, model_dir / "random_forest_aqi.pkl")
(ROOT / "reports").mkdir(exist_ok=True)
pd.Series(metrics, name="value").to_csv(ROOT / "reports/metrics.csv", header=True)
clean_dir = ROOT / "data/interim"
clean_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(clean_dir / "city_day_clean.csv", index=False)
print("Saved model, holdout metrics, and cleaned data.")
